# AdMIRe Multimodal Ranking: Objective Alignment & Training Dynamics

This notebook isolates and evaluates the direct effect of the ranking loss formulation on downstream retrieval performance and generalization behavior. Maintaining the previously derived optimal hyperparameters and core architecture (`film_full`), we contrast standard **Listwise Ranking Optimization** against a **Target-Aware Hybrid Loss** designed to directly penalize Top-1 retrieval misalignment.

Crucially, we track epoch-wise **Training vs. Validation (Dev) Loss trajectories** across all random seeds to visually audit model convergence and identify potential overfitting boundaries.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure project root is accessible
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ranking_experiments.utils import (
    set_seed,
    build_loader,
    build_query_projection,
    forward_scores,
    top1_accuracy,
    ndcg_at_k,
)
from src.config import QueryFusionConfig, get_device
from src.query_fusion import build_query_fusion
from src.candidate_fusion import GatedCandidateFusion
from src.ranker import CosineRanker

print("Libraries and dependencies imported successfully.")

In [ ]:
# Load Modality Cache and Base Configurations
cache_path = project_root / "ranking_experiments/caching/cache/admire_rank_cache.pt"
cache = torch.load(cache_path, map_location="cpu")
train_q_raw = cache["splits"]["train"]["q_raw"]
splits = cache["splits"]
device = get_device()

SEEDS = (42, 100, 150, 200, 250)
SELECTED_ARCHITECTURE = "film_full"
PROJECTION_TYPE = "linear_trainable"
LEARNABLE_LOGIT_SCALE = True
EPOCHS = 25

# Load Global Optimal Parameters from Phase 2 Tuning
best_params_file = project_root / "ranking_experiments/hyperparam_tuning/best_params.json"
if best_params_file.exists():
    with open(best_params_file, "r") as f:
        tuning_meta = json.load(f)
    opt_bs = tuning_meta["best_params"]["batch_size"]
    opt_lr = tuning_meta["best_params"]["learning_rate"]
    print(f"Loaded Global Optimal Hyperparameters -> BS: {opt_bs}, LR: {opt_lr:.6f}")
else:
    opt_bs, opt_lr = 64, 5e-4
    print(f"Fallback Hyperparameters -> BS: {opt_bs}, LR: {opt_lr}")

### Compared Objectives (Loss Functions)

In this ablation study, we evaluate the impact of two distinct objective formulations on multimodal ranking performance:

1. **Pure Listwise Loss ($\mathcal{L}_{listwise}$):**  
   Our standard InfoNCE-derived objective that aligns the predicted similarity score sequence distribution with the ground-truth relevance distribution:
   $$\mathcal{L}_{listwise} = -\sum_{i} P_{gold}(i) \log P_{model}(i)$$

2. **Target-Aware Hybrid Loss ($\mathcal{L}_{hybrid}$):**  
   Designed to directly optimize exact candidate identification (our leading metric, **Top-1 Retrieval Accuracy**), this objective combines the global sequence penalty with a targeted Cross-Entropy loss on the positive candidate index:
   $$\mathcal{L}_{hybrid} = \alpha \cdot \mathcal{L}_{listwise} + (1 - \alpha) \cdot \mathcal{L}_{CE\_Top1}$$
   *(In this configuration, the mixing weight is fixed to $\alpha=0.3$).*

In [ ]:
# Define Objective Loss Operators
def pure_listwise_loss(scores: torch.Tensor, gold_relevance: torch.Tensor, gold_order: torch.Tensor) -> torch.Tensor:
    target_distribution = F.softmax(gold_relevance.float(), dim=-1)
    log_scores = F.log_softmax(scores, dim=-1)
    return -(target_distribution * log_scores).sum(dim=-1).mean()

def target_hybrid_loss(scores: torch.Tensor, gold_relevance: torch.Tensor, gold_order: torch.Tensor, alpha: float = 0.3) -> torch.Tensor:
    target_distribution = F.softmax(gold_relevance.float(), dim=-1)
    log_scores = F.log_softmax(scores, dim=-1)
    l_listwise = -(target_distribution * log_scores).sum(dim=-1).mean()
    l_ce = F.cross_entropy(scores, gold_order[:, 0])
    return alpha * l_listwise + (1.0 - alpha) * l_ce

LOSS_OPERATORS = [
    ("Pure Listwise Loss", pure_listwise_loss),
    ("Hybrid Loss (0.3 LW + 0.7 CE)", target_hybrid_loss)
]

In [ ]:
# Execution Setup Supporting Multi-Split Evaluation and Loss Curve Tracking
class DummyConfig:
    def __init__(self, name):
        self.query_projection = PROJECTION_TYPE
        self.m_embedding_dim = 64
        self.fusion_type = SELECTED_ARCHITECTURE
        self.logit_scale_learnable = LEARNABLE_LOGIT_SCALE
        self.name = name
        self.m_source = "predicted"

def train_epoch_tracking(modules: dict, loader: torch.utils.data.DataLoader, optimizer: torch.optim.Optimizer, device: torch.device, loss_fn) -> float:
    modules["query_projection"].train()
    modules["query_fusion"].train()
    modules["candidate_fusion"].train()
    total_loss, total_examples = 0.0, 0
    for batch in loader:
        gold_relevance = batch["gold_relevance"].to(device)
        gold_order = batch["gold_order"].to(device)
        optimizer.zero_grad(set_to_none=True)
        scores = forward_scores(batch, "predicted", modules["query_projection"], modules["query_fusion"], modules["candidate_fusion"], modules["ranker"], device)
        loss = loss_fn(scores, gold_relevance, gold_order)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * gold_relevance.size(0)
        total_examples += gold_relevance.size(0)
    return total_loss / total_examples

def evaluate_split_tracking(modules: dict, loader: torch.utils.data.DataLoader, device: torch.device, loss_fn) -> dict:
    for mod in modules.values():
        mod.eval()
    losses, top1_vals, ndcg_vals = [], [], []
    with torch.no_grad():
        for batch in loader:
            gold_relevance = batch["gold_relevance"].to(device)
            gold_order = batch["gold_order"].to(device)
            scores = forward_scores(batch, "predicted", modules["query_projection"], modules["query_fusion"], modules["candidate_fusion"], modules["ranker"], device)
            losses.append(loss_fn(scores, gold_relevance, gold_order).item())
            top1_vals.append(top1_accuracy(scores, gold_order))
            ndcg_vals.append(ndcg_at_k(scores, gold_relevance))
    return {"loss": sum(losses) / len(losses), "top1": sum(top1_vals) / len(top1_vals), "ndcg": sum(ndcg_vals) / len(ndcg_vals)}

In [ ]:
# Execute Multi-Seed Optimization Sweeps Tracking Train vs Validation Convergence Curves
summary_records = []
loss_curve_records = []

for loss_label, loss_fn in LOSS_OPERATORS:
    print(f"\n>>> Executing Multi-Seed Evaluation & Curve Tracking for: {loss_label}")
    seed_test_top1, seed_test_ndcg = [], []
    seed_xeval_top1, seed_xeval_ndcg = [], []
    
    # Pre-allocate containers to average epoch-wise curves across seeds
    avg_train_loss_per_epoch = {e: [] for e in range(1, EPOCHS + 1)}
    avg_dev_loss_per_epoch = {e: [] for e in range(1, EPOCHS + 1)}
    
    for seed in tqdm(SEEDS, desc=f"Seeds ({loss_label})"):
        set_seed(seed)
        seed_loaders = {
            "train": build_loader(splits["train"], opt_bs, True, seed),
            "dev": build_loader(splits["dev"], opt_bs, False, seed),
            "test": build_loader(splits["test"], opt_bs, False, seed),
            "xeval": build_loader(splits["xeval"], opt_bs, False, seed),
        }
        
        dummy = DummyConfig(f"s{seed}")
        query_projection, _ = build_query_projection(dummy, train_q_raw, device)
        query_fusion = build_query_fusion(QueryFusionConfig(m_embedding_dim=64, fusion_type=SELECTED_ARCHITECTURE)).to(device)
        candidate_fusion = GatedCandidateFusion().to(device)
        ranker = CosineRanker(learnable=LEARNABLE_LOGIT_SCALE).to(device)
        
        modules = {"query_projection": query_projection, "query_fusion": query_fusion, "candidate_fusion": candidate_fusion, "ranker": ranker}
        params = []
        for mod in modules.values():
            params.extend([p for p in mod.parameters() if p.requires_grad])
        optimizer = torch.optim.AdamW(params, lr=opt_lr)
        
        best_dev_ndcg = -1.0
        best_weights = None
        
        for epoch in range(1, EPOCHS + 1):
            t_loss = train_epoch_tracking(modules, seed_loaders["train"], optimizer, device, loss_fn)
            dev_res = evaluate_split_tracking(modules, seed_loaders["dev"], device, loss_fn)
            
            avg_train_loss_per_epoch[epoch].append(t_loss)
            avg_dev_loss_per_epoch[epoch].append(dev_res["loss"])
            
            # Track optimal checkpoint based on validation set sequential alignment
            if dev_res["ndcg"] > best_dev_ndcg:
                best_dev_ndcg = dev_res["ndcg"]
                best_weights = {k: v.state_dict() for k, v in modules.items()}
                
        # Restore best performing generalized weights prior to Test evaluation
        if best_weights:
            for k, v in modules.items():
                v.load_state_dict(best_weights[k])
                
        test_res = evaluate_split_tracking(modules, seed_loaders["test"], device, loss_fn)
        seed_test_top1.append(test_res["top1"])
        seed_test_ndcg.append(test_res["ndcg"])
        
        xeval_res = evaluate_split_tracking(modules, seed_loaders["xeval"], device, loss_fn)
        seed_xeval_top1.append(xeval_res["top1"])
        seed_xeval_ndcg.append(xeval_res["ndcg"])
        
    # Store trajectory summaries
    summary_records.append({
        "Objective Formulation": loss_label,
        "Test Set Top-1 Accuracy (%)": (sum(seed_test_top1) / len(seed_test_top1)) * 100.0,
        "Test Set nDCG (%)": (sum(seed_test_ndcg) / len(seed_test_ndcg)) * 100.0,
        "Cross-Lingual Top-1 Accuracy (%)": (sum(seed_xeval_top1) / len(seed_xeval_top1)) * 100.0,
        "Cross-Lingual nDCG (%)": (sum(seed_xeval_ndcg) / len(seed_xeval_ndcg)) * 100.0,
    })
    
    # Append cross-seed average epoch values to rendering records
    for epoch in range(1, EPOCHS + 1):
        mean_t_loss = sum(avg_train_loss_per_epoch[epoch]) / len(avg_train_loss_per_epoch[epoch])
        mean_d_loss = sum(avg_dev_loss_per_epoch[epoch]) / len(avg_dev_loss_per_epoch[epoch])
        loss_curve_records.append({"Epoch": epoch, "Loss Value": mean_t_loss, "Split": "Train Loss", "Objective": loss_label})
        loss_curve_records.append({"Epoch": epoch, "Loss Value": mean_d_loss, "Split": "Validation (Dev) Loss", "Objective": loss_label})
        
df_summary = pd.DataFrame(summary_records)
df_curves = pd.DataFrame(loss_curve_records)

In [ ]:
# Render Explicit Held-Out Outcomes Matrix
print("\n=== HELD-OUT EVALUATION MATRIX ===")
display(df_summary.round(2))

# Render Performance Gains as Warning-Free Panels
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left panel: Exact Top-1 Retrieval Accuracy
melted_top1 = df_summary.melt(id_vars=["Objective Formulation"], 
                              value_vars=["Test Set Top-1 Accuracy (%)", "Cross-Lingual Top-1 Accuracy (%)"],
                              var_name="Split", value_name="Top-1 Score (%)")
melted_top1['Split'] = melted_top1['Split'].str.replace(" Set Top-1 Accuracy (%)", "", regex=False).str.replace(" Top-1 Accuracy (%)", "", regex=False)

sns.barplot(data=melted_top1, x="Split", y="Top-1 Score (%)", hue="Objective Formulation", 
            palette="rocket", width=0.5, ax=axes[0])
axes[0].set_title("Exact Top-1 Retrieval Alignment (Held-Out Splits)", fontsize=13, pad=12, fontweight='bold')
axes[0].set_ylabel("Top-1 Accuracy (%)")
axes[0].set_xlabel("")
axes[0].set_ylim(0, 80)

for p in axes[0].patches:
    height = p.get_height()
    if height > 0:
        axes[0].annotate(f'{height:.2f}%', (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom', xytext=(0, 4), textcoords='offset points', fontsize=10, fontweight='bold')

# Right panel: Global Sequence Preservation (nDCG)
melted_ndcg = df_summary.melt(id_vars=["Objective Formulation"], 
                              value_vars=["Test Set nDCG (%)", "Cross-Lingual nDCG (%)"],
                              var_name="Split", value_name="nDCG Score (%)")
melted_ndcg['Split'] = melted_ndcg['Split'].str.replace(" Set nDCG (%)", "", regex=False).str.replace(" nDCG (%)", "", regex=False)

sns.barplot(data=melted_ndcg, x="Split", y="nDCG Score (%)", hue="Objective Formulation", 
            palette="mako", width=0.5, ax=axes[1])
axes[1].set_title("Global Sequence Ranking Quality (nDCG)", fontsize=13, pad=12, fontweight='bold')
axes[1].set_ylabel("nDCG Score (%)")
axes[1].set_xlabel("")
axes[1].set_ylim(60, 100)

for p in axes[1].patches:
    height = p.get_height()
    if height > 0:
        axes[1].annotate(f'{height:.2f}%', (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom', xytext=(0, 4), textcoords='offset points', fontsize=10)

sns.despine()
plt.tight_layout()
plt.show()

### Multi-Seed Convergence Audit: Epoch-Wise Trajectories

Visualizing cross-seed average training and validation loss trajectories to inspect optimization stability and identify points of onset overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

palette = sns.color_palette("husl", 2)

for idx, (loss_label, _) in enumerate(LOSS_OPERATORS):
    ax = axes[idx]
    df_obj = df_curves[df_curves["Objective"] == loss_label]
    
    df_train = df_obj[df_obj["Split"] == "Train Loss"]
    df_dev = df_obj[df_obj["Split"] == "Validation (Dev) Loss"]
    
    ax.plot(df_train["Epoch"], df_train["Loss Value"], label="Train Loss", color=palette[idx], linewidth=2.5, marker="o")
    ax.plot(df_dev["Epoch"], df_dev["Loss Value"], label="Validation (Dev) Loss", color=palette[idx], linewidth=2.0, linestyle="--", marker="s")
    
    # Highlight minimum validation loss
    min_dev = df_dev.loc[df_dev["Loss Value"].idxmin()]
    ax.scatter(min_dev["Epoch"], min_dev["Loss Value"], color="red", s=120, zorder=5, label=f"Min Val Loss ({min_dev['Loss Value']:.3f})")
    
    ax.set_title(f"Optimization Dynamics: {loss_label}", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Training Epoch")
    ax.set_ylabel("Loss Magnitude")
    ax.legend(frameon=True, facecolor="white", framealpha=1.0)
    ax.grid(True, linestyle=":", alpha=0.6)

sns.despine()
plt.suptitle("Epoch-Wise Optimization Trajectories Across Random Seeds", fontsize=16, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()